## 01: Import Libraries
We import all tools needed for training Random Forest.
- pandas and numpy: for data handling
- sklearn: for model training, splitting, and evaluation
- imblearn: for SMOTE to handle class imbalance
- matplotlib and seaborn: for plotting results

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported")

All libraries imported


## 02: Load the Merged Feature Dataset
We load the final merged_features.csv.
This dataset has 2754 windows and 23 features combining
ECG HRV, CNN probability, EEG band power, and respiratory
features.

In [4]:
SAVE_PATH = r"C:\Users\EmaSk\Desktop\sleep-apnea-detection\eman"

df = pd.read_csv(f"{SAVE_PATH}\\merged_features.csv")

print(f"Dataset loaded")
print(f"Shape : {df.shape}")
print(f"Normal: {np.sum(df['label']==0)}")
print(f"Apnea : {np.sum(df['label']==1)}")
print(f"\nFeature columns: {list(df.columns[:-1])}")

Dataset loaded
Shape : (2754, 24)
Normal: 2389
Apnea : 365

Feature columns: ['mean_rr', 'sdnn', 'rmssd', 'mean_hr', 'pnn50', 'lf_hf_ratio', 'cnn_prob', 'delta_power', 'theta_power', 'alpha_power', 'beta_power', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'delta_beta_ratio', 'breathing_rate', 'mean_amplitude', 'std_amplitude', 'mean_peak_dist', 'breath_regularity', 'apnea_index', 'ie_ratio']


## 03: Split Data and Apply SMOTE
We split the data into 80% training and 20% testing using
stratified split to preserve class proportions.
Then we apply SMOTE (Synthetic Minority Oversampling Technique)
only on the training data to balance the apnea class.
This creates synthetic apnea samples so the model doesn't
become biased towards predicting normal all the time.

In [6]:
from imblearn.over_sampling import SMOTE

X = df.drop(columns=['label'])
y = df['label']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE — Train: Normal={np.sum(y_train==0)}, Apnea={np.sum(y_train==1)}")

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"After SMOTE  — Train: Normal={np.sum(y_train_smote==0)}, Apnea={np.sum(y_train_smote==1)}")
print(f"\nTest set (untouched): Normal={np.sum(y_test==0)}, Apnea={np.sum(y_test==1)}")

Before SMOTE — Train: Normal=1911, Apnea=292
After SMOTE  — Train: Normal=1911, Apnea=1911

Test set (untouched): Normal=478, Apnea=73


## 04: Train Random Forest Classifier
We train a Random Forest with 200 trees on the SMOTE-balanced
training data. Random Forest builds many decision trees and
combines their predictions, making it robust and accurate
for tabular feature data like ours.

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest trained")
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Apnea']))

Random Forest trained

Test Accuracy: 0.8875 (88.75%)

Classification Report:
              precision    recall  f1-score   support

      Normal       0.94      0.93      0.93       478
       Apnea       0.57      0.60      0.59        73

    accuracy                           0.89       551
   macro avg       0.76      0.77      0.76       551
weighted avg       0.89      0.89      0.89       551



## 05: Save the Trained Model and Predictions
We save the Random Forest model and its predictions so we
can use them for confusion matrix, ROC curve, and
the ablation study. We also save feature importances which
will help Amna create her top-10 feature bar chart.

In [9]:
import joblib

SAVE_PATH = r"C:\Users\EmaSk\Desktop\sleep-apnea-detection\eman"

# Save model
joblib.dump(rf_model, f"{SAVE_PATH}\\rf_model.pkl")

# Save predictions and test data for Day 7
np.save(f"{SAVE_PATH}\\rf_y_test.npy", y_test.values)
np.save(f"{SAVE_PATH}\\rf_y_pred.npy", y_pred_rf)
np.save(f"{SAVE_PATH}\\rf_y_pred_proba.npy", y_pred_proba_rf)

# Save feature importances
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance_df.to_csv(f"{SAVE_PATH}\\rf_feature_importance.csv", index=False)

print(f"Model and results saved")
print(f"\nTop 10 most important features:")
print(feature_importance_df.head(10).to_string(index=False))

Model and results saved

Top 10 most important features:
          feature  importance
         cnn_prob    0.204931
      theta_power    0.073997
breath_regularity    0.060131
 delta_beta_ratio    0.059047
   mean_amplitude    0.049377
         ie_ratio    0.042090
      delta_power    0.039238
   breathing_rate    0.038925
         rel_beta    0.038067
   mean_peak_dist    0.037044
